# Clustering — Niveau 2 (inter-foyer) — Véritable segmentation des ménages

## C'est ici que la segmentation "des ménages" annoncée par le titre de l'article a lieu

Contrairement au Niveau 1 (qui segmente des *journées* d'un même foyer), ce notebook
segmente les **20 foyers REFIT entre eux**, à partir d'une signature agrégée par foyer.
C'est la brique qui manquait dans le TFC original (mono-foyer UCI).

**Étapes :**
1. Construction des signatures ménages (moyenne des 7 features sur toute la période suivie)
2. Clustering inter-foyer (n=20) — Cycle 1 + Cycle 2, adaptés au petit échantillon
3. Bootstrap de stabilité (n=20 est un échantillon modeste pour un GMM+BIC formel)
4. **Validation externe** avec les métadonnées socio-démographiques réelles de REFIT
   (occupation, type de logement, année de construction, nombre d'appareils) — la pièce la
   plus précieuse de tout ce projet.

**Note de rigueur méthodologique :** avec n=20, un choix de k par BIC/GMM formel est fragile.
Ce notebook calcule ces métriques par souci de transparence, mais la lecture de la
classification hiérarchique (dendrogramme) et le bootstrap de stabilité pèsent davantage
dans la décision finale que le score composite seul — à formuler ainsi dans la Discussion
de l'article.

AUTEUR : ENANGA DANIEL CHRISTIAN


## Étape 0 : Imports

In [ ]:
import os
import sys
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
matplotlib.rcParams['figure.dpi'] = 120

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(BASE_DIR, "src"))

from features import FEATURES_FINAL
from clustering_pipeline import (
    run_cycle1, run_cycle2, select_best_across_algorithms,
    refit_final_model, reorder_labels_by_size,
)

PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed", "refit")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
IMAGES_DIR = os.path.join(BASE_DIR, "images", "clustering_level2")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)

RANDOM_STATE = 42
print("Bibliothèques chargées.")


## Étape 1 : Construction des signatures ménages

Pour chaque foyer, la signature ménage est la **moyenne des 7 features sur l'ensemble de sa
période suivie** (~2 ans). Le foyer UCI (France) n'est volontairement PAS inclus dans ce
clustering inter-foyer : mélanger un foyer français et des foyers britanniques introduirait
un facteur de confusion pays/climat non contrôlé. Il reste disponible comme point de
comparaison qualitatif en discussion.

In [ ]:
level1_path = os.path.join(PROCESSED_DIR, "level1_features_raw.csv")
df_level1_raw = pd.read_csv(level1_path, parse_dates=["date"]) if os.path.exists(level1_path) else pd.DataFrame()

if df_level1_raw.empty:
    print("[ATTENTION] level1_features_raw.csv introuvable -- exécutez d'abord les Notebooks 01 et 02.")
else:
    df_household = df_level1_raw.groupby("house_id")[FEATURES_FINAL].mean().reset_index()
    n_days_per_house = df_level1_raw.groupby("house_id").size().rename("n_days_used")
    df_household = df_household.merge(n_days_per_house, on="house_id")

    # Extraction de l'identifiant numérique REFIT (ex: 'REFIT_12' -> 12) pour la jointure avec les métadonnées
    df_household["house_num"] = df_household["house_id"].str.extract(r'(\d+)').astype(int)

    print(f"Signatures ménages construites pour {df_household.shape[0]} foyers.")
    df_household


## Étape 2 : Chargement des métadonnées ménages (Notebook 01)

In [ ]:
metadata_path = os.path.join(PROCESSED_DIR, "household_metadata_clean.csv")
df_metadata = pd.read_csv(metadata_path) if os.path.exists(metadata_path) else pd.DataFrame()

if not df_metadata.empty and not df_level1_raw.empty:
    df_household = df_household.merge(
        df_metadata, left_on="house_num", right_on="house_id", suffixes=("", "_meta")
    )
    print(f"Jointure signatures <-> métadonnées : {df_household.shape[0]} foyers appariés.")
    df_household[["house_id", "occupancy", "house_type", "bedrooms", "construction_year", "appliances_owned"]]


## Étape 3 : Clustering inter-foyer (n=20) — Cycle 1 + Cycle 2

In [ ]:
if not df_household.empty:
    X_household_raw = df_household[FEATURES_FINAL].values
    household_scaler = StandardScaler()
    X_household = household_scaler.fit_transform(X_household_raw)

    K_RANGE_L2 = range(2, 6)  # k exploré volontairement plus restreint qu'au Niveau 1 (n=20 seulement)

    cycle1_l2 = run_cycle1(X_household, k_range=K_RANGE_L2, random_state=RANDOM_STATE)
    cycle2_l2 = run_cycle2(X_household, k_range=K_RANGE_L2, random_state=RANDOM_STATE)
    best_l2 = select_best_across_algorithms(cycle1_l2, cycle2_l2)

    print(f"Meilleur algorithme (Niveau 2) : {best_l2['best_algorithm']}")
    print(f"  k = {best_l2.get('k')}, Silhouette = {best_l2['silhouette']:.4f}, "
          f"Davies-Bouldin = {best_l2['davies_bouldin']:.4f}, Calinski-Harabasz = {best_l2['calinski_harabasz']:.2f}")

    _, labels_l2 = refit_final_model(X_household, best_l2["best_algorithm"], best_l2, random_state=RANDOM_STATE)
    labels_l2_ordered = reorder_labels_by_size(labels_l2)
    df_household["cluster"] = labels_l2_ordered

    print("\nDistribution des clusters ménages :")
    print(df_household["cluster"].value_counts().sort_index())


## Étape 4 : Dendrogramme (lecture qualitative, complémentaire au score composite)

Avec n=20, la lecture visuelle de la structure hiérarchique est au moins aussi informative
que le choix de k par BIC. Ce dendrogramme est à inclure dans l'article aux côtés du tableau
de métriques.

In [ ]:
if not df_household.empty:
    Z = linkage(X_household, method="ward")
    fig, ax = plt.subplots(figsize=(12, 6))
    dendrogram(Z, labels=df_household["house_id"].values, ax=ax, leaf_rotation=90)
    ax.set_title("Dendrogramme (Ward) — 20 foyers REFIT (signatures agrégées)", fontweight="bold")
    ax.set_ylabel("Distance de Ward")
    plt.tight_layout()
    plt.savefig(os.path.join(IMAGES_DIR, "01_dendrogramme_menages.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("Figure sauvegardée : 01_dendrogramme_menages.png")


## Étape 5 : Bootstrap de stabilité

Pour chaque foyer, on reconstruit sa signature à partir d'un ré-échantillonnage (avec remise)
de ses propres journées, B fois, puis on vérifie si l'affectation au cluster reste stable.
Une faible stabilité pour un foyer donné signale une position "limite" entre deux profils.

In [ ]:
def assign_to_nearest_cluster(x_new_scaled: np.ndarray, X_train_scaled: np.ndarray, labels_train: np.ndarray) -> int:
    """Assigne un point au cluster du centroïde le plus proche (repli
    générique utilisable même pour des modèles sans méthode .predict(),
    comme HAC ou K-Medoids)."""
    centroids = np.array([
        X_train_scaled[labels_train == c].mean(axis=0) for c in np.unique(labels_train) if c >= 0
    ])
    dists = cdist(x_new_scaled.reshape(1, -1), centroids)
    return int(np.argmin(dists))


def bootstrap_stability(df_level1_raw, df_household, scaler, X_household, labels, n_boot=500, random_state=42):
    rng = np.random.default_rng(random_state)
    house_ids = df_household["house_id"].values
    stability_counts = {h: 0 for h in house_ids}

    for b in range(n_boot):
        boot_signature_rows = []
        for house_id in house_ids:
            days = df_level1_raw.loc[df_level1_raw["house_id"] == house_id, FEATURES_FINAL].values
            sample_idx = rng.integers(0, len(days), size=len(days))
            boot_signature_rows.append(days[sample_idx].mean(axis=0))

        X_boot_raw = np.array(boot_signature_rows)
        X_boot_scaled = scaler.transform(X_boot_raw)  # même scaler que la signature originale (point estimate)

        for i, house_id in enumerate(house_ids):
            assigned = assign_to_nearest_cluster(X_boot_scaled[i], X_household, labels)
            if assigned == labels[i]:
                stability_counts[house_id] += 1

    stability_df = pd.DataFrame({
        "house_id": house_ids,
        "original_cluster": labels,
        "stability_pct": [100 * stability_counts[h] / n_boot for h in house_ids],
    }).sort_values("stability_pct")
    return stability_df


if not df_household.empty:
    df_stability = bootstrap_stability(
        df_level1_raw, df_household, household_scaler, X_household, labels_l2_ordered,
        n_boot=500, random_state=RANDOM_STATE,
    )
    print(df_stability.to_string(index=False))
    print(f"\nStabilité moyenne : {df_stability['stability_pct'].mean():.1f}%")


## Étape 6 : Validation externe avec les métadonnées REFIT

**C'est le résultat le plus important de l'article.** On teste si les clusters de ménages
correspondent à des différences réelles d'occupation, de type de logement ou d'âge du bâti.

In [ ]:
if not df_household.empty and not df_metadata.empty:
    print("=" * 70)
    print("VALIDATION EXTERNE — VARIABLES CONTINUES (Kruskal-Wallis)")
    print("=" * 70)

    continuous_vars = ["occupancy", "appliances_owned", "bedrooms", "construction_year"]
    kw_results = []
    for var in continuous_vars:
        groups = [
            df_household.loc[df_household["cluster"] == c, var].dropna().values
            for c in sorted(df_household["cluster"].unique())
        ]
        groups = [g for g in groups if len(g) > 0]
        if len(groups) >= 2:
            stat, p = stats.kruskal(*groups)
            kw_results.append({"variable": var, "H_statistic": stat, "p_value": p})
            print(f"  {var:20s} : H = {stat:.3f}, p = {p:.4f} {'(significatif à 5%)' if p < 0.05 else ''}")

    df_kw = pd.DataFrame(kw_results)

    print("\n" + "=" * 70)
    print("VALIDATION EXTERNE — VARIABLE CATÉGORIELLE (type de logement)")
    print("=" * 70)

    contingency = pd.crosstab(df_household["cluster"], df_household["house_type"])
    print(contingency)

    chi2, p_chi2, dof, _ = stats.chi2_contingency(contingency)
    n = contingency.sum().sum()
    min_dim = min(contingency.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else np.nan
    print(f"\nChi2 = {chi2:.3f}, p = {p_chi2:.4f}, V de Cramér = {cramers_v:.3f}")

    print("\nNote d'interprétation : avec n=20 répartis en plusieurs clusters, la puissance")
    print("statistique de ces tests est limitée -- reporter les tailles d'effet (V de Cramér)")
    print("en plus des p-values, et formuler les conclusions avec la prudence appropriée.")


## Étape 7 : Profils moyens par cluster de ménages

In [ ]:
if not df_household.empty:
    profile_cols = FEATURES_FINAL + ["occupancy", "appliances_owned", "bedrooms", "construction_year"]
    cluster_profiles = df_household.groupby("cluster")[profile_cols].mean().round(2)
    print(cluster_profiles.to_string())

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.heatmap(
        df_household.groupby("cluster")[FEATURES_FINAL].mean(),
        annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax,
    )
    ax.set_title("Profils moyens des clusters de ménages (features brutes)", fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(IMAGES_DIR, "02_profils_clusters_menages.png"), dpi=300, bbox_inches="tight")
    plt.show()


## Étape 8 : Sauvegarde

In [ ]:
if not df_household.empty:
    df_household.to_csv(os.path.join(RESULTS_DIR, "level2_household_signatures.csv"), index=False)
    df_stability.to_csv(os.path.join(RESULTS_DIR, "level2_bootstrap_stability.csv"), index=False)
    if 'df_kw' in dir():
        df_kw.to_csv(os.path.join(RESULTS_DIR, "level2_metadata_validation_continuous.csv"), index=False)
    contingency.to_csv(os.path.join(RESULTS_DIR, "level2_metadata_validation_categorical.csv"))

    print("Fichiers Niveau 2 sauvegardés dans :", RESULTS_DIR)


## Étape 9 : À quoi ressemble une journée "typique" pour chaque type de ménage ?

Le Niveau 1 montrait la forme des régimes *à l'intérieur* d'un foyer. Ici, on répond à une
question différente et complémentaire : **à quoi ressemble une journée moyenne pour un ménage
du cluster 0, comparé à un ménage du cluster 3 ?** Chaque courbe ci-dessous moyenne, pour un
cluster de ménages donné, la journée moyenne de CHAQUE foyer membre (chaque foyer compte pour
1, indépendamment de son nombre de jours suivis, pour éviter qu'un foyer très documenté
écrase les autres).

Cellule autonome (relit les fichiers sur disque).

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed", "refit")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
IMAGES_DIR = os.path.join(BASE_DIR, "images", "clustering_level2")

df_household_l2 = pd.read_csv(os.path.join(RESULTS_DIR, "level2_household_signatures.csv"))


def household_mean_hourly_profile(house_id: str) -> pd.Series | None:
    """Moyenne, sur TOUTE la période suivie d'un foyer, sa consommation par
    heure de la journée (une seule courbe résumant tout le foyer)."""
    num = str(house_id).replace("REFIT_", "")
    path = os.path.join(PROCESSED_DIR, f"house_{num}_clean.csv")
    if not os.path.exists(path):
        return None
    series = pd.read_csv(path, index_col=0, parse_dates=True)["power_kw"]
    return series.groupby(series.index.hour).mean()


if not df_household_l2.empty:
    fig, ax = plt.subplots(figsize=(10, 6))

    for cluster in sorted(df_household_l2["cluster"].unique()):
        house_ids = df_household_l2.loc[df_household_l2["cluster"] == cluster, "house_id"]
        profiles = [household_mean_hourly_profile(h) for h in house_ids]
        profiles = [p for p in profiles if p is not None]
        if not profiles:
            continue
        profile_matrix = pd.concat(profiles, axis=1)  # une colonne par foyer du cluster
        mean_profile = profile_matrix.mean(axis=1)
        std_profile = profile_matrix.std(axis=1)

        ax.plot(mean_profile.index, mean_profile.values, linewidth=2.5,
                label=f"Cluster {cluster} (n={len(profiles)} ménages)")
        ax.fill_between(mean_profile.index, (mean_profile - std_profile).clip(lower=0),
                         mean_profile + std_profile, alpha=0.12)

    ax.set_title("Journée moyenne par type de ménage — Niveau 2", fontweight="bold")
    ax.set_xlabel("Heure de la journée")
    ax.set_ylabel("Puissance moyenne (kW)")
    ax.set_xticks(range(0, 24, 2))
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(IMAGES_DIR, "03_profils_horaires_par_type_menage.png"), dpi=300, bbox_inches="tight")
    plt.show()
    print("Figure sauvegardée : 03_profils_horaires_par_type_menage.png")


## Résumé final

**Fichiers produits :**
- `results/level2_household_signatures.csv` — signatures + cluster de chaque foyer
- `results/level2_bootstrap_stability.csv` — stabilité par foyer
- `results/level2_metadata_validation_continuous.csv` / `_categorical.csv` — tests statistiques
- `images/clustering_level2/01_dendrogramme_menages.png`, `02_profils_clusters_menages.png`,
  `03_profils_horaires_par_type_menage.png`

**Étape suivante :** Notebook `05_Explainability.ipynb` — explicabilité aux deux niveaux
(XGBoost + SHAP + arbre de décision), pour comparer les variables discriminantes entre le
Niveau 1 (régimes journaliers) et le Niveau 2 (typologie de ménages).
